<a href="https://colab.research.google.com/github/rsabarigs-AVR/self-learing/blob/main/Python_Files_Session_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Files and Data Handling
**Seller A's upload day: 19 March 2026, window closes at 6 PM**

How to use this notebook:
1. Run the cells **in order, from top to bottom**.
2. To run a cell, click inside it and press **Shift + Enter** (or click the round play button on its left).
3. Before you run a cell, **guess the output**. Then run it and compare.
4. If you see `FileNotFoundError`, run the **Step 0** cell again (Colab forgets uploaded files when it resets).

## Step 0: Set up the data folder
Run the cell below. A **Choose Files** button appears under it. Click it and pick **data.zip** from your laptop.
When it finishes you should see a list of 4 file names.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()                  # pick data.zip from your laptop
zip_name = list(uploaded.keys())[0]        # works even if the file was renamed "data (1).zip"
zipfile.ZipFile(zip_name).extractall(".")  # creates the folder called data
print(sorted(os.listdir("data")))

**Only if Step 0 did not work** (for example, your laptop unzipped the file and you now have 4 separate files instead of data.zip):
run the cell below, click **Choose Files**, and select **all 4 files at once** (hold Ctrl, or Cmd on a Mac, while clicking).

In [ ]:
from google.colab import files
import os, shutil

os.makedirs("data", exist_ok=True)
for name in files.upload():                 # select all 4 files together
    shutil.move(name, os.path.join("data", name))
print(sorted(os.listdir("data")))

## Topic 1: Read the notes and the product table

### 1.1 Read the reminder note into one piece of text

In [ ]:
with open("data/upload_notes.txt", encoding="utf-8") as f:
    notes = f.read()

print(len(notes), notes.splitlines()[0])

In [ ]:
# See the whole note
print(notes)

### 1.2 Read the product table row by row (`csv.reader`)

In [ ]:
import csv

with open("data/seller_product_codes.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.reader(f))

print(rows[0])                                 # the first line: column names
print(len(rows) - 1, rows[1][2], rows[1][4])   # number of products, first code, first price

### 1.3 Keep only Seller A's rows (`csv.DictReader`)

In [ ]:
import csv

seller_a_rows = []
with open("data/seller_product_codes.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["seller_id"] == "SHOP-A":
            seller_a_rows.append(row)

blanks = sum(1 for r in seller_a_rows if not r["title_raw"].strip())
print(len(seller_a_rows), blanks, seller_a_rows[0]["product_code_raw"])

### 1.4 Find the blank titles in the whole shared table

In [ ]:
import csv

with open("data/seller_product_codes.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

blanks = sum(1 for r in rows if not r["title_raw"].strip())
print(blanks, len(rows))

In [ ]:
# Which rows are blank, and whose are they?
for r in rows:
    if not r["title_raw"].strip():
        print(r["row_id"], r["seller_id"], repr(r["title_raw"]))

## Topic 2: Build clean_export.csv

### 2.1 Build a clean list of Seller A's products

In [ ]:
import csv

clean = []
with open("data/seller_product_codes.csv", newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        if r["seller_id"] == "SHOP-A" and r["title_raw"].strip():
            clean.append({
                "product_code": r["product_code_raw"].strip(),
                "title": r["title_raw"].strip().title(),
                "price_inr": r["price_inr"],
            })

print(len(clean), clean[0])

### 2.2 Write the clean list to a new file (header first, then rows)

In [ ]:
with open("data/clean_export.csv", "w", newline="", encoding="utf-8") as g:
    w = csv.DictWriter(g, fieldnames=["product_code", "title", "price_inr"])
    w.writeheader()
    w.writerows(clean)

print(len(clean), clean[0]["product_code"])

Open the **folder icon** on the left, then the **data** folder. You should now see `clean_export.csv`. (Click the refresh icon if you don't.)

### 2.3 Prove it worked: read the new file back

In [ ]:
with open("data/clean_export.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(rows[0]["product_code"], rows[0]["title"], len(rows))

### 2.4 Hands-on: is it *really* clean?
Look at the printout below. Two problems are left:
- some product codes are lowercase (`tee-0208-bk`)
- one title has two spaces (`Running  Shoes`)

In [ ]:
for row in clean:
    print(row["product_code"], "|", row["title"])

In [ ]:
# YOUR TURN: fix both problems, then save the file again
# Hint 1: "abc".upper() gives "ABC"
# Hint 2: " ".join("Running  Shoes".split()) gives "Running Shoes"
for row in clean:
    row["product_code"] = ...      # make it uppercase
    row["title"] = ...             # remove the extra spaces

# Now copy the 4 "with open ... writerows" lines from step 2.2 here to save again


## Topic 3: JSON files and ready-made helpers

### 3.1 Read the kurta detail card (`json.load`)

In [ ]:
import json

with open("data/kurta_detail.json", encoding="utf-8") as f:
    detail = json.load(f)

print(detail["product_code"], type(detail["price_inr"]), detail["upload_date"])

### 3.2 Hands-on: does the table price match the detail card?

In [ ]:
# YOUR TURN: find the kurta row in clean_export.csv and compare its price with detail["price_inr"]
# Hint: the CSV price is text, so use int(...) before comparing
with open("data/clean_export.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["product_code"] == ...:
            print(...)


### 3.3 Read the payments file: a list of records

In [ ]:
with open("data/transactions.json", encoding="utf-8") as f:
    txns = json.load(f)

print(len(txns), txns[0]["txn_id"], type(txns[0]["amount_inr"]))

In [ ]:
# Look at the first record
print(txns[0])

### 3.4 Count the tags with `Counter`

In [ ]:
from collections import Counter

tags = Counter(t["tag"] for t in txns)
top = tags.most_common(3)
print(top[0][0], top[0][1], len(tags))

In [ ]:
print(tags.most_common())

### 3.5 Stamp the export time with `datetime`

In [ ]:
from datetime import datetime

stamp = datetime(2026, 3, 19, 17, 45).strftime("%Y-%m-%d %H:%M")
with open("data/clean_export.csv", newline="", encoding="utf-8") as f:
    n = sum(1 for _ in csv.DictReader(f))

print(stamp, n)

## Topic 4: Fix a broken script with AI, then test the fix

### 4.1 Run the broken script. It will crash: that is expected.

In [ ]:
import csv

total = 0
with open("data/seller_product_codes.csv", newline="", encoding="utf-8") as f:
    for row in csv.reader(f):
        total += int(row[4])

print(total)

### 4.2 Ask an AI tool for a fix
Copy this prompt into ChatGPT, Claude or Gemini, then paste your code and the error message below it:

> **Task:** fix this Python 3 script that adds up column 4 in data/seller_product_codes.csv.
> **Context:** it crashes with ValueError on the price_inr header row.
> **Format:** return only the corrected code, under ten lines.
> **Constraint:** keep csv.reader and skip the header.

Paste the AI's answer into the next cell, **run it**, and check the total.

In [ ]:
# Paste the AI's fixed code here, then run it.
# Expected total: 79980


### 4.3 Bonus bug: total cashback

In [ ]:
total_cashback = 0
for t in txns:
    total_cashback += t["cashback"]
print(total_cashback)

In [ ]:
# YOUR TURN: ask the AI tool why the cell above crashes, paste its fix here, and run it.


## Closing: check and download the export

In [ ]:
with open("data/clean_export.csv", newline="", encoding="utf-8") as f:
    n = sum(1 for _ in csv.DictReader(f))

print("Seller A export rows:", n)

In [ ]:
# Download the clean file to your laptop
files.download("data/clean_export.csv")